# CNN-Based Fashion Classification — Clean Solution (Polished)

End-to-end FashionMNIST CNN with data augmentation, training, evaluation, and prediction visualization.

**Key concepts:** Conv2d, MaxPool, Dropout, Adam, NLLLoss + LogSoftmax, DataLoader, data augmentation.

**Notes:**
- Uses `NLLLoss` with `LogSoftmax` (equivalent to `CrossEntropyLoss` with raw logits).
- Subsets are used for faster experimentation.
- Always call `model.train()` / `model.eval()` and `torch.no_grad()` appropriately.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Step 1: Dataset with Augmentation

In [ ]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(28, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # scale to [-1, 1]
])

full_train = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
full_test = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

train_subset, _ = random_split(full_train, [10000, len(full_train) - 10000])
test_subset, _ = random_split(full_test, [3000, len(full_test) - 3000])

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=False)

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

## Step 2: CNN Architecture

In [ ]:
class FashionMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),          # 28 -> 14
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),          # 14 -> 7
        )
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10),
            nn.LogSoftmax(dim=1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        return self.classifier(x)

model = FashionMNISTCNN().to(device)
print(model)

## Step 3: Loss & Optimizer

In [ ]:
loss_fn = nn.NLLLoss()  # pairs with LogSoftmax
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.99))

## Step 4: Training Loop

In [ ]:
num_epochs = 10
train_losses = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

## Step 5: Evaluation

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy on test data: {accuracy:.2f}%")

## Step 6: Visualize Predictions

In [ ]:
classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle Boot']

model.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)
with torch.no_grad():
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)

plt.figure(figsize=(12, 12))
for i in range(min(25, len(images))):
    plt.subplot(5, 5, i + 1)
    img = images[i].cpu().squeeze()
    plt.imshow(img, cmap='gray')
    plt.title(f"Pred: {classes[predicted[i]]}\nTrue: {classes[labels[i]]}", fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.show()